GRPRISM2D


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from mpl_toolkits.axes_grid1 import make_axes_locatable  
import tkinter as tk
import math
from tkinter import messagebox, filedialog
import os  
# Desativa os avisos de depreciação da biblioteca matemática da Intel
os.environ["MKL_WARN"] = "0"
os.environ["MKL_DISABLE_FAST_MM"] = "1"

import sys

# =============================================================================
# DICIONÁRIO GLOBAL DE IDIOMAS (INTERNACIONALIZAÇÃO)
# =============================================================================
LANG_DATA = {
    'pt': {
        'title_setup': "GRPRISM2D — Parâmetros Iniciais",
        'title_main': "GRPRISM2D - Painel de Modelagem Prismática",
        'sub_main': "—  Centro Editável {0}x{0} Ativo",
        'import_title': "Importar Dados de Campo (Opcional):",
        'load_btn': "📁 CARREGAR ARQUIVO .TXT",
        'status_none': "Nenhum arquivo inserido (Usando anomalia do corpo de teste)",
        'status_loaded': "✓ {} carregado",
        'label_comp': "Comprimento útil da área (km):",
        'label_prof': "Profundidade máxima (km):",
        'label_dist': "Distância entre as estações (m):",
        'frame_radio': " Seleção da Resolução (Malhas Progressivas) ",
        'btn_start': "🚀 INICIAR MODELAGEM",
        'btn_update': "🔄 ATUALIZAR MODELO",
        'btn_exit': "💾 ENCERRAR E SALVAR",
        'err_validation_title': "Erro de Validação",
        'err_validation_msg': "Por favor, digite apenas valores numéricos positivos.",
        'err_input_title': "Erro de Entrada",
        'err_input_msg': "Certifique-se de digitar apenas números nas células livres.",
        'err_read_title': "Erro de Leitura",
        'err_read_msg': "Não foi possível ler o arquivo txt:\n{}",
        'file_dialog_title': "Selecionar Dado Observado Gravimétrico",
        'matrix_editor_title': " Matrix Editor - Contrastes de Densidade (Colunas {} a {} Editáveis) ",
        'cbar_label': "Contraste de Densidade (g/cm³)",
        'plot_obs': "Dado Observado",
        'plot_calc': "Dado Calculado",
        'plot_res': "RMS",
        'plot_title_1': "Ajuste de Curvas Gravimétricas (Erro RMS Total = {:.3f} mGal)",
        'plot_title_2': "Seção do Modelo de Contrastes de Densidade",
        'plot_xlabel': "Distância (km)",
        'plot_ylabel_1': "Gz (mGal)",
        'plot_ylabel_2': "Profundidade (km)",
        'success_title': "Sucesso",
        'success_msg': "Modelagem salva na pasta:\n'{}'"
    },
    'en': {
        'title_setup': "GRPRISM2D — Initial Parameters",
        'title_main': "GRPRISM2D - Prismatic Modeling Panel",
        'sub_main': "—  Editable Center {0}x{0} Active",
        'import_title': "Import Field Data (Optional):",
        'load_btn': "📁 LOAD .TXT FILE",
        'status_none': "No file uploaded (Using synthetic test body anomaly)",
        'status_loaded': "✓ {} loaded",
        'label_comp': "Useful area length (km):",
        'label_prof': "Maximum depth (km):",
        'label_dist': "Distance between stations (m):",
        'frame_radio': " Resolution Selection (Progressive Grids) ",
        'btn_start': "🚀 START MODELING",
        'btn_update': "🔄 UPDATE MODEL",
        'btn_exit': "💾 CLOSE AND SAVE",
        'err_validation_title': "Validation Error",
        'err_validation_msg': "Please enter positive numerical values only.",
        'err_input_title': "Input Error",
        'err_input_msg': "Make sure to enter only numbers in the editable cells.",
        'err_read_title': "Read Error",
        'err_read_msg': "Could not read the txt file:\n{}",
        'file_dialog_title': "Select Gravimetric Observed Data",
        'matrix_editor_title': " Matrix Editor - Density Contrasts (Columns {} to {} Editable) ",
        'cbar_label': "Density Contrast (g/cm³)",
        'plot_obs': "Observed Data",
        'plot_calc': "Calculated Data",
        'plot_res': "RMS",
        'plot_title_1': "Gravimetric Curve Fitting (Total RMS Error = {:.3f} mGal)",
        'plot_title_2': "Density Contrast Model Cross-Section",
        'plot_xlabel': "Distance (km)",
        'plot_ylabel_1': "Gz (mGal)",
        'plot_ylabel_2': "Depth (km)",
        'success_title': "Success",
        'success_msg': "Modeling saved in folder:\n'{}'"
    }
}

current_lang = 'pt'  # Idioma inicial padrão

def t(key):
    """Função auxiliar curta para tradução de chaves"""
    return LANG_DATA[current_lang][key]

# =============================================================================
# 1. FUNÇÕES DE MODELAGEM FÍSICA
# =============================================================================

def gvertical(vecx, R, xq, z1, z2, t_size):
    gz = np.zeros(len(vecx), dtype=float)
    
    # 1. Definição explícita da Constante Gravitacional no SI
    G = 6.6743e-11  # m³ / (kg * s²)
    
    # 2. Converte o contraste de densidade de g/cm³ para kg/m³ (Ex: 0.67 -> 670 kg/m³)
    R_SI = R * 1000.0
    
    for i in range(len(gz)):   
        if z1 == 0:
            if xq == 0 and vecx[i] == t_size:
                gz[i] = (2*G*R_SI)*(((vecx[i])/2)*math.log((z2**2+vecx[i]**2)/(vecx[i]**2))-z2*(math.atan(0)-math.atan((vecx[i]/z2))))
                
            if xq == 0 and vecx[i] != t_size:
                gz[i] = (2*G*R_SI)*(((vecx[i])/2)*math.log((z2**2+(vecx[i])**2)/((vecx[i])**2))+((vecx[i]-t_size)/2)*math.log(((vecx[i]-t_size)**2)/(z2**2+((vecx[i]-t_size)**2)))-z2*(math.atan(((vecx[i]-t_size)/z2))-math.atan((vecx[i]/z2))))

            if xq != 0 and vecx[i] == xq:
                gz[i] = (2*G*R_SI)*(((-t_size)/2)*math.log((((-t_size)**2)/(z2**2+(-t_size)**2)))-z2*(math.atan(((-t_size)/z2))))

            if xq != 0 and vecx[i] != xq:
                if vecx[i]-xq == t_size:
                    gz[i] = (2*G*R_SI)*(((vecx[i]-xq)/2)*math.log((z2**2+(vecx[i]-xq)**2)/(z1**2+(vecx[i]-xq)**2))-z2*(math.atan(0)-math.atan(((vecx[i]-xq)/z2))))
                else:
                    gz[i] = (2*G*R_SI)*(((vecx[i]-xq)/2)*math.log((z2**2+(vecx[i]-xq)**2)/((vecx[i]-xq)**2))+(((vecx[i]-xq)-t_size)/2)*math.log((((vecx[i]-xq)-t_size)**2)/(z2**2+((vecx[i]-xq)-t_size)**2))-z2*(math.atan((((vecx[i]-xq)-t_size)/z2))-math.atan(((vecx[i]-xq)/z2))))        

        else:
             gz[i] = (2*G*R_SI)*(((vecx[i]-xq)/2)*math.log((z2**2+(vecx[i]-xq)**2)/(z1**2+(vecx[i]-xq)**2))+(((vecx[i]-xq)-t_size)/2)*math.log((z1**2+((vecx[i]-xq)-t_size)**2)/(z2**2+((vecx[i]-xq)-t_size)**2))-z2*(math.atan((((vecx[i]-xq)-t_size)/z2))-math.atan(((vecx[i]-xq)/z2)))+z1*(math.atan((((vecx[i]-xq)-t_size)/z1))-math.atan(((vecx[i]-xq)/z1))))        

    return gz

def curva_gv_automatica(densidades, vecx, xq, z1, z2, t_size):
    curva_final = np.zeros(len(vecx), dtype=float)
    prismas_por_camada = len(xq) 
    contador = 0 
    
    for d in densidades:
        for i, j in enumerate(d):
            idx_camada = contador // prismas_por_camada
            if idx_camada >= len(z1):
                break
            curva_final += gvertical(vecx, j, xq[i], z1[idx_camada], z2[idx_camada], t_size)
            contador += 1

    return curva_final / (10**(-5))


# =============================================================================
# 2. JANELA PRINCIPAL DE MODELAGEM (INTERFACES CONECTADAS)
# =============================================================================

def abrir_janela_principal(comprimento_total, profundidade_max, t_size, opcao_malha, dados_finais_campo):
    global entries_grid, ax1, ax2, canvas, cbar, im, vecx, xq, z1, z2, dado_observado
    global limite_esquerda, limite_direita, n_prismas_x, n_prismas_z, linha_calculado

    mapa_linhas = {1: 5, 2: 6, 3: 7, 4: 8, 5: 9, 6: 10}
    n_prismas_z = mapa_linhas.get(opcao_malha, 10)

    limite_esquerda = 5
    colunas_brancas_centro = n_prismas_z  
    limite_direita = limite_esquerda + colunas_brancas_centro
    n_prismas_x = limite_direita + 5  

    xq = np.array([i * t_size for i in range(n_prismas_x)])
    vecx = xq + (t_size / 2.0)

    espessura_camada = profundidade_max / n_prismas_z
    z1 = np.array([i * espessura_camada for i in range(n_prismas_z)])
    z2 = z1 + espessura_camada

    if dados_finais_campo is not None:
        dist_dados = np.linspace(vecx[0], vecx[-1], len(dados_finais_campo))
        dado_observado = np.interp(vecx, dist_dados, dados_finais_campo)
    else:
        matriz_sintetica_alvo = np.zeros((n_prismas_z, n_prismas_x))
        linha_centro = n_prismas_z // 2
        coluna_centro = limite_esquerda + (colunas_brancas_centro // 2)
        matriz_sintetica_alvo[linha_centro, coluna_centro] = 100.0
        matriz_sintetica_alvo[linha_centro, coluna_centro - 1] = 100.0
        dado_observado = curva_gv_automatica(matriz_sintetica_alvo, vecx, xq, z1, z2, t_size)

    main_window = tk.Tk()
    main_window.title(t('title_main'))
    main_window.state('zoomed') if os.name == 'nt' else main_window.geometry(
        "{0}x{1}+0+0".format(main_window.winfo_screenwidth(), main_window.winfo_screenheight())
    )

    main_window.rowconfigure(1, weight=1) 
    main_window.columnconfigure(0, weight=1)

    # Banner Superior
    frame_topo = tk.Frame(main_window, bg="#1e293b", height=65) 
    frame_topo.grid(row=0, column=0, sticky="ew")
    frame_topo.pack_propagate(False)

    canvas_logo = tk.Canvas(frame_topo, width=60, height=60, bg="#1e293b", bd=0, highlightthickness=0)
    canvas_logo.pack(side=tk.LEFT, padx=(20, 10), pady=2)
    canvas_logo.create_line(5, 8, 25, 15, 40, 5, 55, 10, fill="#ef4444", width=2, smooth=True)  
    canvas_logo.create_line(5, 14, 20, 7, 40, 14, 55, 6, fill="#3b82f6", width=2, smooth=True)  
    pontos_corpo = [15,22, 20,19, 30,18, 40,20, 48,23, 45,28, 35,30, 22,29, 16,26]
    canvas_logo.create_polygon(pontos_corpo, fill="#854d0e", outline="") 
    canvas_logo.create_rectangle(8, 31, 52, 34, fill="#475569", outline="")
    canvas_logo.create_line(30,34, 20,38, 40,43, 20,48, 40,53, 30,57, fill="#ffffff", width=2) 
    canvas_logo.create_rectangle(5, 57, 55, 60, fill="#475569", outline="")

    label_texto_logo = tk.Label(frame_topo, text="GRPRISM2D", font=("Helvetica", 20, "bold"), fg="#f8fafc", bg="#1e293b")
    label_texto_logo.pack(side=tk.LEFT, anchor="w", padx=(5, 0))

    label_subtitulo = tk.Label(frame_topo, text=t('sub_main').format(colunas_brancas_centro), font=("Helvetica", 17), fg="#94a3b8", bg="#1e293b")
    label_subtitulo.pack(side=tk.LEFT, padx=10, anchor="w", pady=(4, 0))

    # Painel dos Gráficos com Alinhamento Rígido de Colorbar
    frame_graficos = tk.Frame(main_window)
    frame_graficos.grid(row=1, column=0, sticky="nsew", padx=10, pady=5)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 4.5))
    matriz_inicial = np.zeros((n_prismas_z, n_prismas_x))
    im = ax2.imshow(matriz_inicial, cmap='jet', aspect='auto', extent=[0, (n_prismas_x*t_size)/1000, profundidade_max/1000, 0])

    divider = make_axes_locatable(ax2)
    cax = divider.append_axes("right", size="3%", pad=0.15)
    cbar = fig.colorbar(im, cax=cax, orientation='vertical')
    cbar.set_label(t('cbar_label'), fontsize=10)

    fig.subplots_adjust(left=0.08, right=0.90, top=0.90, bottom=0.12, hspace=0.35)

    canvas = FigureCanvasTkAgg(fig, master=frame_graficos)
    canvas.get_tk_widget().pack(side=tk.TOP, fill=tk.BOTH, expand=True)

    def obter_matriz_densidades():
        matriz = np.zeros((n_prismas_z, n_prismas_x))
        for r in range(n_prismas_z):
            for c in range(n_prismas_x):
                val = entries_grid[r][c].get()
                matriz[r, c] = float(val) if val != "" else 0.0
        return matriz

    def atualizar_modelo():
            global im, canvas, dado_observado, linha_calculado
            try:
                if dados_finais_campo is not None:
                    dist_dados = np.linspace(vecx[0], vecx[-1], len(dados_finais_campo))
                    dado_observado = np.interp(vecx, dist_dados, dados_finais_campo)
                    
                    # Garante o shift para o dado real alinhar verticalmente no zero
                    dado_observado = dado_observado - dado_observado[0]

                densidades_malha = obter_matriz_densidades()
                dado_calculado = curva_gv_automatica(densidades_malha, vecx, xq, z1, z2, t_size)
                
                erro_residuo = dado_observado - dado_calculado
                erro_rms = np.sqrt(np.mean(erro_residuo ** 2))
                
                # --- TRUQUE DO ZOOM PARCIAL (CÁLCULO DOS LIMITES) ---
                largura_coluna_km = t_size / 1000.0
                x_min_visual = 2 * largura_coluna_km  # 5.0 km
                x_max_visual = 17 * largura_coluna_km # 42.5 km
                
                # --- O SEGREDO DO ALINHAMENTO HORIZONTAL DOS DADOS ---
                # Fatiamos o vetor de distâncias do cálculo para bater com o fatiamento da matriz
                vecx_fatiado_km = (vecx / 1000.0)[2:17]
                dado_observado_fatiado = dado_observado[2:17]
                dado_calculado_fatiado = dado_calculado[2:17]
                erro_residuo_fatiado = erro_residuo[2:17]
                
                # --- RECONSTRUÇÃO DO GRÁFICO SUPERIOR (CURVAS) ---
                ax1.clear() 
                # Agora plotamos usando os vetores fatiados nas mesmas posições km reais!
                ax1.plot(vecx_fatiado_km, dado_observado_fatiado, 'ko', label=t('plot_obs'), markersize=4)
                linha_calculado, = ax1.plot(vecx_fatiado_km, dado_calculado_fatiado, 'k-', label=t('plot_calc'), linewidth=1.5)
                ax1.plot(vecx_fatiado_km, erro_residuo_fatiado, 'r--', label=t('plot_res'), linewidth=1.0)
                
                ax1.set_title(t('plot_title_1').format(erro_rms))
                ax1.set_ylabel(t('plot_ylabel_1'))
                ax1.grid(True, linestyle=":")
                ax1.legend(loc='upper right', fontsize=9)
                
                # Fixa o limite horizontal do gráfico superior
                ax1.set_xlim(x_min_visual, x_max_visual)
                
                # --- RECONSTRUÇÃO DO GRÁFICO INFERIOR (SEÇÃO GEOLÓGICA) ---
                ax2.clear()
                matriz_fatiada = densidades_malha[:, 2:17]
                
                im = ax2.imshow(matriz_fatiada, cmap='jet', aspect='auto',
                                extent=[x_min_visual, x_max_visual, profundidade_max / 1000.0, 0])
                
                ax2.set_title(t('plot_title_2'))
                ax2.set_xlabel(t('plot_xlabel'))
                ax2.set_ylabel(t('plot_ylabel_2'))
                
                im.set_data(matriz_fatiada)
                
                # Ajusta a escala de cores dinamicamente baseado nas densidades digitadas
                min_atual = np.min(densidades_malha)
                max_atual = np.max(densidades_malha)
                if min_atual == 0.0 and max_atual == 0.0:
                    im.set_clim(vmin=-0.05, vmax=0.7)
                else:
                    im.set_clim(vmin=min_atual, vmax=max_atual)
                    
                cbar.update_normal(im)
                canvas.draw()
                
            except ValueError:
                messagebox.showerror(t('err_input_title'), t('err_input_msg'))

    # Alternador de idioma na tela principal
    def alternar_idioma_main():
        global current_lang
        current_lang = 'en' if current_lang == 'pt' else 'pt'
        
        # Redefine títulos textuais das janelas e frames
        main_window.title(t('title_main'))
        label_subtitulo.config(text=t('sub_main').format(colunas_brancas_centro))
        frame_container_matrix.config(text=t('matrix_editor_title').format(limite_esquerda+1, limite_direita))
        btn_atualizar.config(text=t('btn_update'))
        btn_encerrar.config(text=t('btn_exit'))
        cbar.set_label(t('cbar_label'), fontsize=10)
        
        # Força atualização dos termos dentro do Matplotlib
        atualizar_modelo()

    btn_lang_main = tk.Button(frame_topo, text="🇺🇸 EN / 🇧🇷 PT", command=alternar_idioma_main, bg="#334155", fg="white", font=("Arial", 9, "bold"), bd=0, padx=8)
    btn_lang_main.pack(side=tk.RIGHT, padx=20, pady=18)

    # Geração dos campos numéricos da matriz
    frame_container_matrix = tk.LabelFrame(main_window, text=t('matrix_editor_title').format(limite_esquerda+1, limite_direita), font=("Arial", 10, "bold"))
    frame_container_matrix.grid(row=2, column=0, sticky="ew", padx=10, pady=5)
    frame_container_matrix.columnconfigure(0, weight=1)

    canvas_scroll = tk.Canvas(frame_container_matrix, height=(n_prismas_z * 28) + 10, highlightthickness=0)
    scrollbar_x = tk.Scrollbar(frame_container_matrix, orient="horizontal", command=canvas_scroll.xview)
    frame_matrix = tk.Frame(canvas_scroll)

    frame_matrix.bind("<Configure>", lambda e: canvas_scroll.configure(scrollregion=canvas_scroll.bbox("all")))
    canvas_window = canvas_scroll.create_window((0, 0), window=frame_matrix, anchor="nw")

    def ajustar_largura_janela(event):
        if event.width > frame_matrix.winfo_reqwidth():
            canvas_scroll.itemconfig(canvas_window, width=event.width)
    canvas_scroll.bind("<Configure>", ajustar_largura_janela)

    canvas_scroll.configure(xscrollcommand=scrollbar_x.set)
    canvas_scroll.grid(row=0, column=0, sticky="ew")
    scrollbar_x.grid(row=1, column=0, sticky="ew")

    entries_grid = []
    for c in range(n_prismas_x):
        frame_matrix.columnconfigure(c, weight=1, uniform="equal")

    for r in range(n_prismas_z):
        row_entries = []
        for c in range(n_prismas_x):
            if c < limite_esquerda or c >= limite_direita:
                en = tk.Entry(frame_matrix, width=2, bg="black", fg="white", justify="center", font=("Arial", 9))
                en.insert(0, "0.0")
                en.config(state="readonly") 
            else:
                en = tk.Entry(frame_matrix, width=2, bg="white", fg="red", justify="center", font=("Arial", 9))
                en.insert(0, "0.0") 
                
            en.grid(row=r, column=c, padx=1, pady=2, sticky="nsew")
            row_entries.append(en)
        entries_grid.append(row_entries)

    def encerrar_modelagem():
            # AVISO CRÍTICO PRO PYTHON: 'main_window' vem da função de cima!
            nonlocal main_window  
            global linha_calculado
            
            nome_pasta = "GRPRISM2D-Resultados"
            os.makedirs(nome_pasta, exist_ok=True)
            densidades_malha = obter_matriz_densidades()
            np.savetxt(os.path.join(nome_pasta, "matriz_densidades_final.txt"), densidades_malha, fmt='%.3f', delimiter='\t')
            try:
                dado_modelado = linha_calculado.get_ydata()
            except NameError:
                dado_modelado = curva_gv_automatica(densidades_malha, vecx, xq, z1, z2, t_size)
                
            np.savetxt(os.path.join(nome_pasta, "dado_modelado_final.txt"), dado_modelado, fmt='%.4f', delimiter='\t')
            fig.savefig(os.path.join(nome_pasta, "modelo_gravimetrico_final.png"), dpi=600,bbox_inches='tight', pad_inches=0.05)
            
            # Mude a linha do messagebox para esta versão com icon='info' explícito:
            messagebox.showinfo(
                t('success_title'), 
                t('success_msg').format(nome_pasta),
                icon='info' # Força o sistema a usar o ícone padrão e ignora o fantasma do Jupyter
            )            
            # --- FECHAMENTO BLINDADO ---
            plt.close('all')           # Fecha os plots do matplotlib
            main_window.withdraw()     # Agora o Python sabe exatamente quem ocultar!
            main_window.quit()         # Para o loop de eventos
            main_window.destroy()      # Destrói a janela
            
            os._exit(0)                # Desliga a célula do Jupyter instantaneamente

    # Botões Inferiores
    frame_botoes = tk.Frame(main_window)
    frame_botoes.grid(row=3, column=0, sticky="ew", pady=10)

    btn_atualizar = tk.Button(frame_botoes, text=t('btn_update'), command=atualizar_modelo, bg="#4CAF50", fg="black", font=("Arial", 10, "bold"), padx=15, pady=5)
    btn_atualizar.pack(side=tk.LEFT, padx=30)

    btn_encerrar = tk.Button(frame_botoes, text=t('btn_exit'), command=encerrar_modelagem, bg="#f44336", fg="black", font=("Arial", 10, "bold"), padx=15, pady=5)
    btn_encerrar.pack(side=tk.RIGHT, padx=30)

    atualizar_modelo()
    main_window.mainloop()


# =============================================================================
# 3. INTERFACE INICIAL DE SETUP
# =============================================================================

dados_txt_globais = None
nome_arquivo_atual = ""

def carregar_arquivo_dados():
    global dados_txt_globais, nome_arquivo_atual
    caminho_arquivo = filedialog.askopenfilename(
        title=t('file_dialog_title'),
        filetypes=[("Arquivos de Texto (*.txt)", "*.txt"), ("Todos os arquivos", "*.*")]
    )
    if caminho_arquivo:
        try:
            with open(caminho_arquivo, 'r', encoding='utf-8', errors='ignore') as f:
                linhas = f.readlines()
            
            linhas_corrigidas = [linha.replace(',', '.') for linha in linhas]
            dados = np.loadtxt(linhas_corrigidas)
            
            if dados.ndim > 1:
                dados = dados[:, -1]
                
            dados_txt_globais = dados
            nome_arquivo_atual = os.path.basename(caminho_arquivo)
            lbl_status_arquivo.config(text=t('status_loaded').format(nome_arquivo_atual), fg="#4CAF50")
        except Exception as e:
            messagebox.showerror(t('err_read_title'), t('err_read_msg').format(str(e)))

def processar_setup_inicial():
    try:
        comp_km = float(entry_comp.get())
        prof_km = float(entry_prof.get())
        dist_m = float(entry_dist.get())
        opcao_idx = var_malha.get()
        
        if comp_km <= 0 or prof_km <= 0 or dist_m <= 0:
            raise ValueError
            
        setup_window.destroy()
        abrir_janela_principal(comp_km * 1000.0, prof_km * 1000.0, dist_m, opcao_idx, dados_txt_globais)
        
    except ValueError:
        messagebox.showerror(t('err_validation_title'), t('err_validation_msg'))

# Inicialização da tela de Setup
setup_window = tk.Tk()
setup_window.title(t('title_setup'))

largura_janela = 560
altura_janela = 620 
largura_tela = setup_window.winfo_screenwidth()
altura_tela = setup_window.winfo_screenheight()
pos_x = int((largura_tela / 2) - (largura_janela / 2))
pos_y = int((altura_tela / 2) - (altura_janela / 2))

setup_window.geometry(f"{largura_janela}x{altura_janela}+{pos_x}+{pos_y}")
setup_window.resizable(False, False)
setup_window.configure(bg="#0f172a")

# Banner Superior Incorporado
frame_topo_setup = tk.Frame(setup_window, bg="#1e293b", height=65) 
frame_topo_setup.pack(side=tk.TOP, fill=tk.X)
frame_topo_setup.pack_propagate(False)

canvas_logo_setup = tk.Canvas(frame_topo_setup, width=60, height=60, bg="#1e293b", bd=0, highlightthickness=0)
canvas_logo_setup.pack(side=tk.LEFT, padx=(20, 10), pady=2)
canvas_logo_setup.create_line(5, 8, 25, 15, 40, 5, 55, 10, fill="#ef4444", width=2, smooth=True)  
canvas_logo_setup.create_line(5, 14, 20, 7, 40, 14, 55, 6, fill="#3b82f6", width=2, smooth=True)  
pontos_corpo_setup = [15,22, 20,19, 30,18, 40,20, 48,23, 45,28, 35,30, 22,29, 16,26]
canvas_logo_setup.create_polygon(pontos_corpo_setup, fill="#854d0e", outline="") 
canvas_logo_setup.create_rectangle(8, 31, 52, 34, fill="#475569", outline="")
canvas_logo_setup.create_line(30,34, 20,38, 40,43, 20,48, 40,53, 30,57, fill="#ffffff", width=2) 
canvas_logo_setup.create_rectangle(5, 57, 55, 60, fill="#475569", outline="")

label_texto_logo_setup = tk.Label(frame_topo_setup, text="GRPRISM2D", font=("Helvetica", 20, "bold"), fg="#f8fafc", bg="#1e293b")
label_texto_logo_setup.pack(side=tk.LEFT, anchor="w", padx=(5, 0))

# Função de alternância dinâmica do menu de Setup
def alternar_idioma_setup():
    global current_lang
    current_lang = 'en' if current_lang == 'pt' else 'pt'
    
    # Atualiza Widgets
    setup_window.title(t('title_setup'))
    lbl_import_title.config(text=t('import_title'))
    btn_arquivo.config(text=t('load_btn'))
    
    if dados_txt_globais is None:
        lbl_status_arquivo.config(text=t('status_none'))
    else:
        lbl_status_arquivo.config(text=t('status_loaded').format(nome_arquivo_atual))
        
    lbl_comp.config(text=t('label_comp'))
    lbl_prof.config(text=t('label_prof'))
    lbl_dist.config(text=t('label_dist'))
    frame_radio.config(text=t('frame_radio'))
    btn_start.config(text=t('btn_start'))

# Botão Seletor no Canto Superior Direito
btn_lang_setup = tk.Button(frame_topo_setup, text="🇺🇸 EN / 🇧🇷 PT", command=alternar_idioma_setup, bg="white", fg="black", font=("Arial", 9, "bold"), bd=0, padx=8)
btn_lang_setup.pack(side=tk.RIGHT, padx=20, pady=18)

# Seção de Entrada de Arquivo TXT de Campo
frame_arquivo = tk.Frame(setup_window, bg="#1e293b", bd=1, relief="solid")
frame_arquivo.pack(fill=tk.X, padx=35, pady=15, ipady=8)

lbl_import_title = tk.Label(frame_arquivo, text=t('import_title'), font=("Arial", 9, "bold"), bg="#1e293b", fg="#38bdf8")
lbl_import_title.pack(pady=2)

btn_arquivo = tk.Button(frame_arquivo, text=t('load_btn'), command=carregar_arquivo_dados, bg="#3b82f6", fg="black", font=("Arial", 9, "bold"), cursor="hand2")
btn_arquivo.pack(pady=4)

lbl_status_arquivo = tk.Label(frame_arquivo, text=t('status_none'), font=("Arial", 8, "italic"), bg="#1e293b", fg="#94a3b8")
lbl_status_arquivo.pack()

# Container das caixas de entrada físicas
frame_inputs = tk.Frame(setup_window, bg="#0f172a")
frame_inputs.pack(pady=5)

lbl_comp = tk.Label(frame_inputs, text=t('label_comp'), font=("Arial", 10, "bold"), bg="#0f172a", fg="#94a3b8")
lbl_comp.grid(row=0, column=0, sticky="w", pady=4, padx=10)
entry_comp = tk.Entry(frame_inputs, width=12, font=("Arial", 10, "bold"), justify="center", bg="#1e293b", fg="#f8fafc", insertbackground="white")
entry_comp.insert(0, "1.5") 
entry_comp.grid(row=0, column=1, pady=4, padx=10)

lbl_prof = tk.Label(frame_inputs, text=t('label_prof'), font=("Arial", 10, "bold"), bg="#0f172a", fg="#94a3b8")
lbl_prof.grid(row=1, column=0, sticky="w", pady=4, padx=10)
entry_prof = tk.Entry(frame_inputs, width=12, font=("Arial", 10, "bold"), justify="center", bg="#1e293b", fg="#f8fafc", insertbackground="white")
entry_prof.insert(0, "1.0")
entry_prof.grid(row=1, column=1, pady=4, padx=10)

lbl_dist = tk.Label(frame_inputs, text=t('label_dist'), font=("Arial", 10, "bold"), bg="#0f172a", fg="#94a3b8")
lbl_dist.grid(row=2, column=0, sticky="w", pady=4, padx=10)
entry_dist = tk.Entry(frame_inputs, width=12, font=("Arial", 10, "bold"), justify="center", bg="#1e293b", fg="#f8fafc", insertbackground="white")
entry_dist.insert(0, "100")
entry_dist.grid(row=2, column=1, pady=4, padx=10)

# Container para a escolha das malhas de refinamento
frame_radio = tk.LabelFrame(setup_window, text=t('frame_radio'), font=("Arial", 10, "bold"), bg="#1e293b", fg="#38bdf8", labelanchor="n", bd=1, relief="solid")
frame_radio.pack(fill=tk.X, padx=35, pady=10, ipady=5)

var_malha = tk.IntVar(value=1)

opcoes_texto = [
    "Mesh 5x19  —  25 prismas",
    "Mesh 6x16  —  36 prismas",
    "Mesh 7x17  —  49 prismas",
    "Mesh 8x18  —  64 prismas",
    "Mesh 9x19  —  81 prismas",
    "Mesh 10x20 —  100 prismas"
]

for idx, txt in enumerate(opcoes_texto, start=1):
    rb = tk.Radiobutton(frame_radio, text=txt, variable=var_malha, value=idx, font=("Arial", 9, "bold"), bg="#1e293b", fg="#94a3b8", selectcolor="#0f172a", activebackground="#1e293b", anchor="w")
    rb.pack(fill=tk.X, padx=20, pady=2)

btn_start = tk.Button(setup_window, text=t('btn_start'), command=processar_setup_inicial, bg="#4CAF50", fg="black", font=("Arial", 11, "bold"), padx=25, pady=8, cursor="hand2", bd=0)
btn_start.pack(pady=(15, 10))

setup_window.mainloop()# =============================================================================
# 3. INTERFACE INICIAL DE SETUP (VERSÃO ULTRA-LEVE DETRAVADA)
# =============================================================================

dados_txt_globais = None
nome_arquivo_atual = ""
lista_radio_buttons = [] # Lista global para rastrear os radiobuttons e limpá-los

def carregar_arquivo_dados():
    global dados_txt_globais, nome_arquivo_atual
    caminho_arquivo = filedialog.askopenfilename(
        title=t('file_dialog_title'),
        filetypes=[("Arquivos de Texto (*.txt)", "*.txt"), ("Todos os arquivos", "*.*")]
    )
    if caminho_arquivo:
        try:
            with open(caminho_arquivo, 'r', encoding='utf-8', errors='ignore') as f:
                linhas = f.readlines()
            
            linhas_corrigidas = [linha.replace(',', '.') for linha in linhas]
            dados = np.loadtxt(linhas_corrigidas)
            
            if dados.ndim > 1:
                dados = dados[:, -1]
                
            dados_txt_globais = dados
            nome_arquivo_atual = os.path.basename(caminho_arquivo)
            lbl_status_arquivo.config(text=t('status_loaded').format(nome_arquivo_atual), fg="#4CAF50")
        except Exception as e:
            messagebox.showerror(t('err_read_title'), t('err_read_msg').format(str(e)))

def processar_setup_inicial():
    try:
        comp_km = float(entry_comp.get())
        prof_km = float(entry_prof.get())
        dist_m = float(entry_dist.get())
        opcao_idx = var_malha.get()
        
        if comp_km <= 0 or prof_km <= 0 or dist_m <= 0:
            raise ValueError
            
        setup_window.destroy()
        abrir_janela_principal(comp_km * 1000.0, prof_km * 1000.0, dist_m, opcao_idx, dados_txt_globais)
        
    except ValueError:
        messagebox.showerror(t('err_validation_title'), t('err_validation_msg'))

def atualizar_textos_malha():
    """Atualiza o texto dos Radiobuttons sem recriá-los na memória"""
    # Lista de chaves dependendo do idioma atual
    if current_lang == 'pt':
        opcoes = [
            "Malha 5x19  —  25 prismas",
            "Malha 6x16  —  36 prismas",
            "Malha 7x17  —  49 prismas",
            "Malha 8x18  —  64 prismas",
            "Malha 9x19  —  81 prismas",
            "Malha 10x20 —  100 prismas"
        ]
    else:
        opcoes = [
            "Grid 5x19  —  25 prisms",
            "Grid 6x16  —  36 prisms",
            "Grid 7x17  —  49 prisms",
            "Grid 8x18  —  64 prisms",
            "Grid 9x19  —  81 prisms",
            "Grid 10x20 —  100 prisms"
        ]
    
    # Aplica os novos textos nos botões existentes de forma leve
    for idx, rb in enumerate(lista_radio_buttons):
        rb.config(text=opcoes[idx])

# Inicialização da tela de Setup
setup_window = tk.Tk()
setup_window.title(t('title_setup'))

largura_janela = 560
altura_janela = 620 
largura_tela = setup_window.winfo_screenwidth()
altura_tela = setup_window.winfo_screenheight()
pos_x = int((largura_tela / 2) - (largura_janela / 2))
pos_y = int((altura_tela / 2) - (altura_janela / 2))

setup_window.geometry(f"{largura_janela}x{altura_janela}+{pos_x}+{pos_y}")
setup_window.resizable(False, False)
setup_window.configure(bg="#0f172a")

# Banner Superior Incorporado
frame_topo_setup = tk.Frame(setup_window, bg="#1e293b", height=65) 
frame_topo_setup.pack(side=tk.TOP, fill=tk.X)
frame_topo_setup.pack_propagate(False)

canvas_logo_setup = tk.Canvas(frame_topo_setup, width=60, height=60, bg="#1e293b", bd=0, highlightthickness=0)
canvas_logo_setup.pack(side=tk.LEFT, padx=(20, 10), pady=2)
canvas_logo_setup.create_line(5, 8, 25, 15, 40, 5, 55, 10, fill="#ef4444", width=2, smooth=True)  
canvas_logo_setup.create_line(5, 14, 20, 7, 40, 14, 55, 6, fill="#3b82f6", width=2, smooth=True)  
pontos_corpo_setup = [15,22, 20,19, 30,18, 40,20, 48,23, 45,28, 35,30, 22,29, 16,26]
canvas_logo_setup.create_polygon(pontos_corpo_setup, fill="#854d0e", outline="") 
canvas_logo_setup.create_rectangle(8, 31, 52, 34, fill="#475569", outline="")
canvas_logo_setup.create_line(30,34, 20,38, 40,43, 20,48, 40,53, 30,57, fill="#ffffff", width=2) 
canvas_logo_setup.create_rectangle(5, 57, 55, 60, fill="#475569", outline="")

label_texto_logo_setup = tk.Label(frame_topo_setup, text="GRPRISM2D", font=("Helvetica", 20, "bold"), fg="#f8fafc", bg="#1e293b")
label_texto_logo_setup.pack(side=tk.LEFT, anchor="w", padx=(5, 0))

# Função de alternância dinâmica do menu de Setup (CORRIGIDA)
def alternar_idioma_setup():
    global current_lang
    current_lang = 'en' if current_lang == 'pt' else 'pt'
    
    # Atualiza Widgets de texto fixo
    setup_window.title(t('title_setup'))
    lbl_import_title.config(text=t('import_title'))
    btn_arquivo.config(text=t('load_btn'))
    
    if dados_txt_globais is None:
        lbl_status_arquivo.config(text=t('status_none'))
    else:
        lbl_status_arquivo.config(text=t('status_loaded').format(nome_arquivo_atual))
        
    lbl_comp.config(text=t('label_comp'))
    lbl_prof.config(text=t('label_prof'))
    lbl_dist.config(text=t('label_dist'))
    frame_radio.config(text=t('frame_radio'))
    btn_start.config(text=t('btn_start'))
    
    # Executa a troca limpa dos textos dos radiobuttons
    atualizar_textos_malha()

# Botão Seletor no Canto Superior Direito
btn_lang_setup = tk.Button(frame_topo_setup, text="🇺🇸 EN / 🇧🇷 PT", command=alternar_idioma_setup, bg="white", fg="black", font=("Arial", 9, "bold"), bd=0, padx=8)
btn_lang_setup.pack(side=tk.RIGHT, padx=20, pady=18)

# Seção de Entrada de Arquivo TXT de Campo
frame_arquivo = tk.Frame(setup_window, bg="#1e293b", bd=1, relief="solid")
frame_arquivo.pack(fill=tk.X, padx=35, pady=15, ipady=8)

lbl_import_title = tk.Label(frame_arquivo, text=t('import_title'), font=("Arial", 9, "bold"), bg="#1e293b", fg="#38bdf8")
lbl_import_title.pack(pady=2)

btn_arquivo = tk.Button(frame_arquivo, text=t('load_btn'), command=carregar_arquivo_dados, bg="#3b82f6", fg="black", font=("Arial", 9, "bold"), cursor="hand2")
btn_arquivo.pack(pady=4)

lbl_status_arquivo = tk.Label(frame_arquivo, text=t('status_none'), font=("Arial", 8, "italic"), bg="#1e293b", fg="#94a3b8")
lbl_status_arquivo.pack()

# Container das caixas de entrada físicas
frame_inputs = tk.Frame(setup_window, bg="#0f172a")
frame_inputs.pack(pady=5)

lbl_comp = tk.Label(frame_inputs, text=t('label_comp'), font=("Arial", 10, "bold"), bg="#0f172a", fg="#94a3b8")
lbl_comp.grid(row=0, column=0, sticky="w", pady=4, padx=10)
entry_comp = tk.Entry(frame_inputs, width=12, font=("Arial", 10, "bold"), justify="center", bg="#1e293b", fg="#f8fafc", insertbackground="white")
entry_comp.insert(0, "1.5") 
entry_comp.grid(row=0, column=1, pady=4, padx=10)

lbl_prof = tk.Label(frame_inputs, text=t('label_prof'), font=("Arial", 10, "bold"), bg="#0f172a", fg="#94a3b8")
lbl_prof.grid(row=1, column=0, sticky="w", pady=4, padx=10)
entry_prof = tk.Entry(frame_inputs, width=12, font=("Arial", 10, "bold"), justify="center", bg="#1e293b", fg="#f8fafc", insertbackground="white")
entry_prof.insert(0, "1.0")
entry_prof.grid(row=1, column=1, pady=4, padx=10)

lbl_dist = tk.Label(frame_inputs, text=t('label_dist'), font=("Arial", 10, "bold"), bg="#0f172a", fg="#94a3b8")
lbl_dist.grid(row=2, column=0, sticky="w", pady=4, padx=10)
entry_dist = tk.Entry(frame_inputs, width=12, font=("Arial", 10, "bold"), justify="center", bg="#1e293b", fg="#f8fafc", insertbackground="white")
entry_dist.insert(0, "100")
entry_dist.grid(row=2, column=1, pady=4, padx=10)

# Container para a escolha das malhas de refinamento
frame_radio = tk.LabelFrame(setup_window, text=t('frame_radio'), font=("Arial", 10, "bold"), bg="#1e293b", fg="#38bdf8", labelanchor="n", bd=1, relief="solid")
frame_radio.pack(fill=tk.X, padx=35, pady=10, ipady=5)

var_malha = tk.IntVar(value=1)

# Construção inicial fixa dos RadioButtons
opcoes_iniciais = [
    "Malha 5x19  —  25 prismas",
    "Malha 6x16  —  36 prismas",
    "Malha 7x17  —  49 prismas",
    "Malha 8x18  —  64 prismas",
    "Malha 9x19  —  81 prismas",
    "Malha 10x20 —  100 prismas"
]

lista_radio_buttons = []
for idx, txt in enumerate(opcoes_iniciais, start=1):
    rb = tk.Radiobutton(frame_radio, text=txt, variable=var_malha, value=idx, font=("Arial", 9, "bold"), bg="#1e293b", fg="#94a3b8", selectcolor="#0f172a", activebackground="#1e293b", anchor="w")
    rb.pack(fill=tk.X, padx=20, pady=2)
    lista_radio_buttons.append(rb) # Guardamos a referência do botão na lista

btn_start = tk.Button(setup_window, text=t('btn_start'), command=processar_setup_inicial, bg="#4CAF50", fg="black", font=("Arial", 11, "bold"), padx=25, pady=8, cursor="hand2", bd=0)
btn_start.pack(pady=(15, 10))

setup_window.mainloop()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


: 